# Autonomous intracellular AND-gate — an MHC-free self-destruct for the dark core (RUNG-23)

**CPU + Census (~15–25 min). No GPU.** Shriya's original concept, the un-crowded route.

The immune route dies on the MHC-dark core (~4%). This tests whether a synthetic **gene circuit inside the cell** — *signal A AND signal B → self-destruct*, no MHC needed — can fire in tumour cells but ~zero vital normal cells. Tests all pairs of 8 intracellular programs (proliferation, MYC, glycolysis, hypoxia, WNT, …) on real single-cell data, with worst-donor vital-leak safety.

**If a clean AND-pair exists → a buildable, MHC-independent, AND-specific autonomous therapy for the cells nothing else reaches.** If not → an honest negative bounding the autonomous route. **Set Runtime → CPU.**

In [ ]:
#@title Cell 1 — clone / pull
import os
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — run log + VALIDATE (selftest)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG=new_log('rung23_autonomous', repo_dir='.')
rc=run_logged([sys.executable,'-u','scripts/49_autonomous_andgate.py','selftest'], RUNLOG)
print('[CELL 2]', '✓ validated' if rc==0 else f'✗ FAILED ({rc}) — stop')

In [ ]:
#@title Cell 3 — (Census) run the autonomous AND-gate scan  [~15-25 min]
import os, sys, importlib.util
from scripts.runlog import run_logged
try:
    from google.colab import drive; drive.mount('/content/drive')
    cd='/content/drive/MyDrive/cancer-recon'; os.makedirs(cd, exist_ok=True)
    os.environ['RUNG23_CACHE']=cd
except Exception as e: print('no Drive (', type(e).__name__, ')')
for pk,pn in [('cellxgene_census','cellxgene-census'),('scanpy','scanpy')]:
    if importlib.util.find_spec(pk) is None:
        run_logged([sys.executable,'-m','pip','install','-q',pn], RUNLOG, label=f'pip {pn}')
rc=run_logged([sys.executable,'-u','scripts/49_autonomous_andgate.py','run'], RUNLOG)
from IPython.display import Image, display
import json
p='runs/rung23_autonomous/rung23_autonomous'
if os.path.exists(p+'.png'): display(Image(p+'.png'))
if os.path.exists(p+'.json'):
    d=json.load(open(p+'.json'))
    print('HEADLINE:', d['HEADLINE'])
    print('\nsafe&effective gates:', json.dumps(d['safe_effective_gates'], indent=2)[:1500])
    print('\ntop ranked:', json.dumps(d['all_gates_ranked'][:8], indent=2))
print('[CELL 3]', '✓ done' if rc==0 else f'(exit {rc}; re-run to resume)')

In [ ]:
#@title Cell 4 — bundle zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem=os.path.basename(str(RUNLOG)).replace('rung23_autonomous_','').replace('.log','')
b=f'/content/rung23_autonomous_{stem}.zip'
ps=sorted(glob.glob('runs/rung23_autonomous/*.png')+glob.glob('runs/rung23_autonomous/*.json')+[str(RUNLOG)])
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 4] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')